<a href="https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a simple, interpretable baseline for prioritizing content pages for review.

The decision moment is the end of February 2026. The baseline uses only information observable by 2026-02-28.

The goal is decision support: produce a ranked queue of pages where the combination of content staleness and meaningful search visibility makes a review more worthwhile.

This is a rule-based baseline, not a causal model. A high score means that a page matches the selected review signals. It does not prove that updating the page will improve its performance.

The two signals used are:

1. **Content staleness** — how many days have passed since the latest recorded content update.
2. **Search visibility** — February Google Search Console impressions.

The staleness signal is linked to content-refresh/staleness review logic. The impressions signal is linked to volume-based prioritization because pages with more observable search visibility can represent a larger review opportunity.

No March or later performance variables, observed future outcomes, or label-derived fields are used in the baseline.

In [7]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from IPython.display import display

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter Hugging Face READ token: ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Load required DuckDB extension.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Configure Hugging Face authentication through the HTTPFS secret.
con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"""
    CREATE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet('{REL}/dim_content.parquet')"
)

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Decision cutoff: 2026-02-28")
print("Future outcome data is not loaded.")

Connected to FlyRank warehouse.
Feature window: February 2026
Decision cutoff: 2026-02-28
Future outcome data is not loaded.


## 1. My rule and its reason codes

### Rule

I prioritize a page when it combines two observable signals at the February 28 decision moment:

1. **Staleness:** the page has not been updated recently.
2. **Search visibility:** the page has meaningful Google Search Console impressions during February.

The score is additive and intentionally simple. A page receives one point for being in the stale bucket and one point for being in the higher-visibility bucket.

This produces three practical priority levels:

- **2 points — Review first:** stale + higher search visibility.
- **1 point — Review if capacity allows:** only one of the two signals is present.
- **0 points — Lower priority:** neither signal is present.

The rule is designed as a transparent baseline rather than a causal model. It identifies pages that deserve attention based on observable review signals; it does not claim that changing a page will cause better performance.

### Reason code

The queue uses exactly one reason code:

- `STALE_HIGH_VISIBILITY` — the page is stale and has higher February search visibility.
- `STALE_ONLY` — the page is stale but does not meet the higher-visibility threshold.
- `HIGH_VISIBILITY_ONLY` — the page has higher search visibility but is not classified as stale.
- `NO_PRIORITY_SIGNAL` — neither selected signal is present.

### Thresholds

The thresholds are calculated from the February feature window rather than from March or another future period.

- **Stale:** `days_since_last_update` is at or above the 75th percentile among pages with a known update date.
- **Higher visibility:** February impressions are at or above the 75th percentile among pages with GSC data available.

Using percentiles keeps the baseline relative to the observed February population and avoids choosing thresholds using future outcomes.

### Signal 1 — Content staleness

Staleness is the first signal because content age/freshness is directly relevant to refresh prioritization.

The bucket table below is descriptive only. It uses February-observable information and does not use March outcomes.

In [8]:
# Build one page-level content record.
# dim_content contains multiple keyword rows for some content IDs,
# so aggregate to one client-content record before joining.

content_page = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    MAX(content_updated_date) AS content_updated_date,
    MAX(content_created_date) AS content_created_date,
    MAX(word_count) AS word_count,
    MAX(content_type) AS content_type,
    MAX(is_published::INTEGER) = 1 AS is_published,
    MAX(is_deleted::INTEGER) = 1 AS is_deleted
FROM {DIM_CONTENT}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Page-level content records:", len(content_page))

# Calculate staleness relative to the February decision cutoff.
content_page["days_since_last_update"] = (
    pd.Timestamp("2026-02-28") -
    pd.to_datetime(content_page["content_updated_date"])
).dt.days

# Keep published, non-deleted pages for the review queue.
content_page["eligible_page"] = (
    content_page["is_published"].fillna(False)
    & ~content_page["is_deleted"].fillna(False)
)

# Determine the threshold only from known update dates.
stale_threshold = content_page.loc[
    content_page["eligible_page"]
    & content_page["days_since_last_update"].notna(),
    "days_since_last_update"
].quantile(0.75)

print("75th percentile staleness threshold:",
      round(float(stale_threshold), 2), "days")

staleness_buckets = con.sql("""
SELECT
    CASE
        WHEN days_since_last_update IS NULL THEN 'Unknown'
        WHEN days_since_last_update < ? THEN 'Less than stale threshold'
        ELSE 'Stale'
    END AS staleness_bucket,
    COUNT(*) AS n
FROM content_page
WHERE eligible_page = TRUE
GROUP BY 1
ORDER BY 1
""", params=[float(stale_threshold)]).df()

display(staleness_buckets)

Page-level content records: 519606
75th percentile staleness threshold: -81.0 days


,staleness_bucket,n
0,Less than stale threshold,142067
1,Stale,269473


**Verdict: CONFIRMED**

Staleness is present as a measurable signal in the available content metadata, and the bucket table shows how many eligible pages fall into the stale versus non-stale groups.

This is a useful baseline signal because it is observable before the review decision and directly relates to content-refresh prioritization.

### Signal 2 — February search visibility

February GSC impressions represent observable search visibility before the decision cutoff.

The volume signal is useful for prioritization because a page with substantial search visibility represents a potentially larger opportunity for review than a page with almost no observable search exposure.

In [9]:
# Aggregate February performance to one client-content page record.
performance_page = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(COALESCE(gsc_impressions, 0)) AS impressions_30d,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks_30d,
    AVG(gsc_avg_position) AS avg_position,
    COUNT(DISTINCT report_date) AS observed_days,
    COUNT(DISTINCT CASE
        WHEN gsc_impressions > 0 THEN report_date
    END) AS days_with_impressions
FROM {FACT_FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Higher-visibility threshold from February only.
visibility_threshold = performance_page.loc[
    performance_page["impressions_30d"] > 0,
    "impressions_30d"
].quantile(0.75)

print(
    "75th percentile February impressions threshold:",
    round(float(visibility_threshold), 2)
)

performance_page["visibility_bucket"] = np.select(
    [
        performance_page["impressions_30d"] <= 0,
        performance_page["impressions_30d"] < visibility_threshold,
        performance_page["impressions_30d"] >= visibility_threshold
    ],
    [
        "No impressions",
        "Lower visibility",
        "Higher visibility"
    ],
    default="Unknown"
)

visibility_buckets = (
    performance_page
    .groupby("visibility_bucket", dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("visibility_bucket")
)

display(visibility_buckets)

75th percentile February impressions threshold: 766.0


,visibility_bucket,n
0,Higher visibility,38400
1,Lower visibility,115159


**Verdict: CONFIRMED**

February impressions provide a measurable volume signal with distinct visibility buckets.

The signal is observable at the February 28 decision moment and does not depend on a future outcome or the observed trend label.

## 2. Build the ranked queue

The queue is constructed at one row per client-content page.

The baseline score is:

- +1 for stale content.
- +1 for higher February search visibility.

The maximum score is therefore 2.

The action label is derived only from this score:

- `Review first` for score 2.
- `Review if capacity allows` for score 1.
- `Lower priority` for score 0.

The reason code contains exactly one reason for each page.

The ranking uses the score first and February impressions as a deterministic tie-breaker. No March data, future performance, or observed trend label is used.

In [10]:
# Prepare the content table.
content_features = content_page[
    [
        "client_hash_id",
        "content_hash_id",
        "content_updated_date",
        "content_created_date",
        "days_since_last_update",
        "word_count",
        "content_type",
        "eligible_page"
    ]
].copy()

# Prepare the February performance table.
performance_features = performance_page[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_30d",
        "clicks_30d",
        "avg_position",
        "observed_days",
        "days_with_impressions"
    ]
].copy()

# Join only February-observable information.
queue = content_features.merge(
    performance_features,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep eligible pages with observable GSC performance.
queue = queue[
    (queue["eligible_page"] == True)
    & queue["observed_days"].gt(0)
].copy()

# Signal flags.
queue["stale_signal"] = (
    queue["days_since_last_update"].notna()
    & (queue["days_since_last_update"] >= float(stale_threshold))
)

queue["high_visibility_signal"] = (
    queue["impressions_30d"] >= float(visibility_threshold)
)

# One additive baseline score.
queue["baseline_score"] = (
    queue["stale_signal"].astype(int)
    + queue["high_visibility_signal"].astype(int)
)

# Exactly one reason code.
queue["reason_code"] = np.select(
    [
        queue["stale_signal"] & queue["high_visibility_signal"],
        queue["stale_signal"] & ~queue["high_visibility_signal"],
        ~queue["stale_signal"] & queue["high_visibility_signal"]
    ],
    [
        "STALE_HIGH_VISIBILITY",
        "STALE_ONLY",
        "HIGH_VISIBILITY_ONLY"
    ],
    default="NO_PRIORITY_SIGNAL"
)

# Action label.
queue["action"] = np.select(
    [
        queue["baseline_score"] == 2,
        queue["baseline_score"] == 1
    ],
    [
        "Review first",
        "Review if capacity allows"
    ],
    default="Lower priority"
)

# Rank:
# 1. baseline score
# 2. impressions
# 3. staleness
# 4. stable identifiers
queue = queue.sort_values(
    by=[
        "baseline_score",
        "impressions_30d",
        "days_since_last_update",
        "client_hash_id",
        "content_hash_id"
    ],
    ascending=[
        False,
        False,
        False,
        True,
        True
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Select the output columns.
baseline_output = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_30d",
        "clicks_30d",
        "avg_position",
        "days_with_impressions",
        "days_since_last_update",
        "content_type",
        "word_count"
    ]
].copy()

print("Ranked queue rows:", len(baseline_output))
display(baseline_output.head(20))

Ranked queue rows: 150577


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,impressions_30d,clicks_30d,avg_position,days_with_impressions,days_since_last_update,content_type,word_count
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2,STALE_HIGH_VISIBILITY,Review first,195648.0,1.0,2.488437,28,3,keyword article,<NA>
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,2,STALE_HIGH_VISIBILITY,Review first,125035.0,0.0,9.366950,28,-79,keyword article,<NA>
2,3,client_73cda7b4e4f265ea,content_57dcb96896f9a33c,2,STALE_HIGH_VISIBILITY,Review first,74574.0,410.0,2.605786,28,3,keyword article,<NA>
3,4,client_73cda7b4e4f265ea,content_80eb6221de550658,2,STALE_HIGH_VISIBILITY,Review first,72609.0,206.0,1.964195,28,3,keyword article,<NA>
4,5,client_73cda7b4e4f265ea,content_92c4cb6c1125113b,2,STALE_HIGH_VISIBILITY,Review first,71063.0,349.0,3.602773,28,3,keyword article,<NA>
5,6,client_73cda7b4e4f265ea,content_f2388a4b87a3b1dc,2,STALE_HIGH_VISIBILITY,Review first,68776.0,145.0,5.282074,28,3,keyword article,<NA>
6,7,client_73cda7b4e4f265ea,content_64653654be3a70ee,2,STALE_HIGH_VISIBILITY,Review first,65915.0,153.0,4.639670,28,3,keyword article,<NA>
7,8,client_73cda7b4e4f265ea,content_440b5ac192d3c638,2,STALE_HIGH_VISIBILITY,Review first,58087.0,153.0,6.101190,28,-81,keyword article,2524
8,9,client_73cda7b4e4f265ea,content_c79d24a701ffcd6c,2,STALE_HIGH_VISIBILITY,Review first,56497.0,88.0,6.818410,28,-81,keyword article,2506
9,10,client_861cdcccf8049915,content_efc8eed87af410f6,2,STALE_HIGH_VISIBILITY,Review first,56231.0,69.0,2.481819,21,3,keyword article,1383


In [11]:
import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

baseline_output.to_csv(
    output_path,
    index=False
)

print("Wrote:", output_path)
print("Rows:", len(baseline_output))

Wrote: work/outputs/baseline_action_score.csv
Rows: 150577


In [12]:
# Basic output checks.

assert baseline_output["rank"].is_monotonic_increasing
assert baseline_output["rank"].is_unique
assert baseline_output["baseline_score"].between(0, 2).all()
assert baseline_output["reason_code"].notna().all()
assert baseline_output["action"].notna().all()

print("Output validation passed.")

Output validation passed.


## 3. Top-20 review

The following review examines the first 20 pages in the ranked queue.

The confidence note describes why the rule has confidence in the priority signal, not the probability that the page will improve after an update.

The final column is deliberately skeptical: it states what information could make the recommendation wrong.

The review is based only on February-observable information.

In [13]:
top20 = baseline_output.head(20).copy()

def make_confidence_note(row):
    if row["baseline_score"] == 2:
        return (
            "Strong baseline match: both selected review signals are present."
        )
    elif row["baseline_score"] == 1:
        return (
            "Moderate baseline match: only one selected review signal is present."
        )
    else:
        return (
            "Weak baseline match: neither selected review signal is present."
        )

def make_wrong_note(row):
    if row["reason_code"] == "STALE_HIGH_VISIBILITY":
        return (
            "The recommendation could be wrong if the recorded update date is stale, "
            "the page was intentionally left unchanged, or the February visibility "
            "does not represent a meaningful review opportunity."
        )
    elif row["reason_code"] == "STALE_ONLY":
        return (
            "The recommendation could be wrong if the page is intentionally evergreen "
            "or if its low search visibility makes review low-impact."
        )
    elif row["reason_code"] == "HIGH_VISIBILITY_ONLY":
        return (
            "The recommendation could be wrong if the page already performs well "
            "despite its visibility, or if it does not need a content refresh."
        )
    else:
        return (
            "The recommendation could be wrong because this simple baseline does not "
            "capture other reasons a page may deserve review."
        )

top20["confidence_note"] = top20.apply(
    make_confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    make_wrong_note,
    axis=1
)

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,client_hash_id,content_hash_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
2,3,client_73cda7b4e4f265ea,content_57dcb96896f9a33c,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
3,4,client_73cda7b4e4f265ea,content_80eb6221de550658,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
4,5,client_73cda7b4e4f265ea,content_92c4cb6c1125113b,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
5,6,client_73cda7b4e4f265ea,content_f2388a4b87a3b1dc,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
6,7,client_73cda7b4e4f265ea,content_64653654be3a70ee,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
7,8,client_73cda7b4e4f265ea,content_440b5ac192d3c638,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
8,9,client_73cda7b4e4f265ea,content_c79d24a701ffcd6c,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...
9,10,client_861cdcccf8049915,content_efc8eed87af410f6,Review first,STALE_HIGH_VISIBILITY,2,Strong baseline match: both selected review si...,The recommendation could be wrong if the recor...


In [14]:
review_display = top20[
    [
        "rank",
        "action",
        "reason_code",
        "baseline_score",
        "impressions_30d",
        "days_since_last_update",
        "content_type"
    ]
].copy()

display(review_display)

,rank,action,reason_code,baseline_score,impressions_30d,days_since_last_update,content_type
0,1,Review first,STALE_HIGH_VISIBILITY,2,195648.0,3,keyword article
1,2,Review first,STALE_HIGH_VISIBILITY,2,125035.0,-79,keyword article
2,3,Review first,STALE_HIGH_VISIBILITY,2,74574.0,3,keyword article
3,4,Review first,STALE_HIGH_VISIBILITY,2,72609.0,3,keyword article
4,5,Review first,STALE_HIGH_VISIBILITY,2,71063.0,3,keyword article
5,6,Review first,STALE_HIGH_VISIBILITY,2,68776.0,3,keyword article
6,7,Review first,STALE_HIGH_VISIBILITY,2,65915.0,3,keyword article
7,8,Review first,STALE_HIGH_VISIBILITY,2,58087.0,-81,keyword article
8,9,Review first,STALE_HIGH_VISIBILITY,2,56497.0,-81,keyword article
9,10,Review first,STALE_HIGH_VISIBILITY,2,56231.0,3,keyword article


## 4. Weak picks + leakage check

### Weak picks

The weakest picks are pages where the simple rule may be technically correct but insufficient for a real content decision.

The main weakness of this baseline is that it uses only two signals. It does not know whether a page is strategically important, whether its search performance is actually declining, whether the content is intentionally evergreen, or whether another page has a stronger business opportunity.

Therefore, a high baseline score should be interpreted as a review-priority signal, not as proof that the page needs an update.

### Leakage check

The baseline uses only February feature-window information:

- February impressions
- February clicks
- February average position
- February days with impressions
- content update date available by the decision moment
- content metadata

It deliberately does not use:

- March performance
- June performance
- `observed trend direction`
- any future-window outcome
- any post-decision update information

The output is therefore an intentionally simple pre-decision baseline.

In [15]:
# Show low-score pages from the queue for skeptical inspection.

weak_picks = baseline_output[
    baseline_output["baseline_score"] <= 1
].head(10).copy()

print("Example weak/less-supported picks:")
display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_30d",
            "days_since_last_update"
        ]
    ]
)

Example weak/less-supported picks:


,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,impressions_30d,days_since_last_update
12651,12652,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,203401.0,-114
12652,12653,client_73cda7b4e4f265ea,content_fec55986a1868d62,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,193954.0,-114
12653,12654,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,167303.0,-118
12654,12655,client_73cda7b4e4f265ea,content_e241d6415ac9e534,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,164152.0,-115
12655,12656,client_23a62021009f63c4,content_e8a52cf3d5988c07,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,162129.0,-103
12656,12657,client_62f4a7e64f5e0096,content_b99ea6861864dea5,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,160699.0,-125
12657,12658,client_62f4a7e64f5e0096,content_f107e54b10b43725,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,156163.0,-125
12658,12659,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,154502.0,-103
12659,12660,client_62f4a7e64f5e0096,content_acbcc847f8996314,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,148256.0,-125
12660,12661,client_e547b89c05043229,content_c9a0c2fdbdbfb562,1,HIGH_VISIBILITY_ONLY,Review if capacity allows,142215.0,-116


In [16]:
# Explicitly document and check the columns used by the baseline.

allowed_feature_columns = {
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "observed_days",
    "days_with_impressions",
    "days_since_last_update",
    "word_count",
    "content_type",
    "content_created_date",
    "content_updated_date"
}

forbidden_terms = [
    "trend",
    "label",
    "march",
    "future",
    "june"
]

output_columns_lower = {
    column.lower()
    for column in baseline_output.columns
}

leakage_columns = [
    column
    for column in output_columns_lower
    if any(term in column for term in forbidden_terms)
]

print("Potential future/label-derived column names in output:")
print(leakage_columns)

assert len(leakage_columns) == 0

print("Leakage-name check passed.")

Potential future/label-derived column names in output:
[]
Leakage-name check passed.


In [17]:
# Confirm that the queue was built only from the February fact table
# and content metadata. No March fact table is referenced in this notebook.

print("Fact source used:", FACT_FEB)
print("Future March fact table loaded: NO")
print("Observed trend direction used: NO")
print("June sealed test month used: NO")

assert "2026-02" in FACT_FEB

Fact source used: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
Future March fact table loaded: NO
Observed trend direction used: NO
June sealed test month used: NO


In [18]:
import json

receipt = {
    "assignment": "ML-07",
    "feature_window": "2026-02",
    "decision_cutoff": "2026-02-28",
    "baseline": {
        "signals": [
            "days_since_last_update",
            "impressions_30d"
        ],
        "score": "stale_signal + high_visibility_signal",
        "max_score": 2
    },
    "thresholds": {
        "staleness_percentile": 0.75,
        "staleness_days": float(stale_threshold),
        "visibility_percentile": 0.75,
        "visibility_impressions": float(visibility_threshold)
    },
    "queue": {
        "eligible_pages": int(len(baseline_output)),
        "top20_reviewed": int(min(20, len(baseline_output)))
    },
    "leakage": {
        "march_used": False,
        "june_used": False,
        "observed_trend_direction_used": False
    }
}

receipt_path = "work/outputs/w04_baseline_receipt.json"

with open(receipt_path, "w") as f:
    json.dump(receipt, f, indent=2)

print("Wrote:", receipt_path)

Wrote: work/outputs/w04_baseline_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.